In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

validation = pd.read_parquet(
    PROCESSED_DIR / "validation.parquet"
)

product_catalog = pd.read_parquet(
    PROCESSED_DIR / "product_catalog.parquet"
)

product_embeddings = np.load(
    RESULTS_DIR / "product_embeddings_bge_small.npy"
)

product_uids = np.load(
    RESULTS_DIR / "product_uids.npy"
)

In [2]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
queries = validation["search_term"].unique()
query_embeddings = model.encode(
    queries.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

In [4]:
def retrieve_top_k(
    query_embeddings,
    product_embeddings,
    product_uids,
    k=100,
    batch_size=32,
):
    results = []

    for start in range(0, len(query_embeddings), batch_size):
        batch = query_embeddings[start:start + batch_size]

        scores = batch @ product_embeddings.T

        top_indices = np.argpartition(
            scores,
            -k,
            axis=1
        )[:, -k:]

        top_scores = np.take_along_axis(
            scores,
            top_indices,
            axis=1
        )

        # Sort the top-k results by similarity
        order = np.argsort(
            top_scores,
            axis=1
        )[:, ::-1]

        top_indices = np.take_along_axis(
            top_indices,
            order,
            axis=1
        )

        top_scores = np.take_along_axis(
            top_scores,
            order,
            axis=1
        )

        for indices, scores_row in zip(top_indices, top_scores):
            results.append({
                "product_uids": product_uids[indices],
                "scores": scores_row,
            })

    return results

In [5]:
retrieved_results = retrieve_top_k(
    query_embeddings,
    product_embeddings,
    product_uids,
    k=100,
)

In [6]:
retrieved_dict = {
    query: result["product_uids"]
    for query, result in zip(
        queries,
        retrieved_results
    )
}

In [7]:
relevant_counts = (
    validation[validation["relevance"] >= 2]
    .groupby("search_term")["product_uid"]
    .nunique()
)

print("Queries with >=1 relevant product:", len(relevant_counts))
print("Queries with >=2 relevant products:", (relevant_counts >= 2).sum())
print("Queries with >=3 relevant products:", (relevant_counts >= 3).sum())

Queries with >=1 relevant product: 1031
Queries with >=2 relevant products: 995
Queries with >=3 relevant products: 891


In [8]:
def calculate_recall_at_k(
    validation,
    retrieved_dict,
    k=10,
):
    recalls = []

    for query, group in validation.groupby("search_term"):
        relevant = set(
            group.loc[
                group["relevance"] >= 2,
                "product_uid"
            ]
        )

        if not relevant:
            continue

        retrieved = set(
            retrieved_dict[query][:k]
        )

        recall = len(relevant & retrieved) / len(relevant)

        recalls.append(recall)

    return np.mean(recalls)

In [9]:
for k in [10, 20, 50, 100]:
    recall = calculate_recall_at_k(
        validation,
        retrieved_dict,
        k=k,
    )

    print(f"Recall@{k}: {recall:.6f}")

Recall@10: 0.221024
Recall@20: 0.314979
Recall@50: 0.450136
Recall@100: 0.550313


In [10]:
def calculate_ndcg_at_k(
    validation,
    retrieved_dict,
    k=10,
):
    ndcgs = []

    for query, group in validation.groupby("search_term"):

        relevance_dict = (
            group
            .set_index("product_uid")["relevance"]
            .to_dict()
        )

        retrieved = retrieved_dict[query][:k]

        ranked_relevances = [
            relevance_dict.get(product_uid, 0)
            for product_uid in retrieved
        ]

        # DCG
        dcg = sum(
            rel / np.log2(rank + 2)
            for rank, rel in enumerate(ranked_relevances)
        )

        # Ideal ranking
        ideal_relevances = sorted(
            relevance_dict.values(),
            reverse=True
        )[:k]

        # IDCG
        idcg = sum(
            rel / np.log2(rank + 2)
            for rank, rel in enumerate(ideal_relevances)
        )

        if idcg == 0:
            continue

        ndcgs.append(dcg / idcg)

    return np.mean(ndcgs)

In [11]:
for k in [10, 20, 50, 100]:
    ndcg = calculate_ndcg_at_k(
        validation,
        retrieved_dict,
        k=k,
    )

    print(f"NDCG@{k}: {ndcg:.6f}")

NDCG@10: 0.193822
NDCG@20: 0.233592
NDCG@50: 0.282300
NDCG@100: 0.312400


In [12]:
import pickle

for query, retrieved_uids in retrieved_dict.items():
    retrieved_dict[query] = retrieved_uids.tolist()

with open(PROCESSED_DIR / "dense_retrieved_dict.pkl", "wb") as f:
    pickle.dump(retrieved_dict, f)

In [13]:
retrieved_dict

{'deck over': [186042,
  121372,
  104312,
  128684,
  118626,
  224175,
  116816,
  219511,
  202562,
  218220,
  118388,
  192460,
  158385,
  132865,
  183356,
  119659,
  202625,
  123905,
  104727,
  152352,
  101629,
  141444,
  121637,
  146204,
  194572,
  125497,
  152093,
  106784,
  137450,
  197006,
  101050,
  101795,
  175019,
  122193,
  107229,
  181869,
  111275,
  103356,
  164963,
  209957,
  190410,
  155284,
  207259,
  209445,
  165377,
  206003,
  223109,
  211475,
  105449,
  100002,
  157930,
  110043,
  147197,
  126900,
  177747,
  109643,
  111978,
  161495,
  198399,
  125996,
  209670,
  125535,
  213491,
  136870,
  147200,
  163307,
  130898,
  118656,
  129208,
  176391,
  207425,
  160187,
  170178,
  154922,
  189681,
  198489,
  137610,
  193270,
  122556,
  155725,
  202666,
  221866,
  138804,
  111452,
  125318,
  214781,
  207112,
  131970,
  211898,
  197299,
  126085,
  167285,
  195510,
  196636,
  117660,
  155460,
  142561,
  158348,
  10694